# Ch.2 — Collaborative Filtering _(Exercise)_

> **The story.** In **1994**, Paul Resnick and colleagues coined the term "collaborative filtering" in the **GroupLens** paper — a system where Usenet readers collaboratively filtered articles by rating them, and those ratings were used to recommend articles to similar readers. The insight was radical: you don't need to understand _what_ an article (or movie) is about. You only need to know _who liked it_. If Alice and Bob vote identically on 50 articles, Alice will probably like what Bob rated highly but hasn't yet seen. Seven years later, **Sarwar, Karypis, Konstan and Riedl** (2001) published the paper Amazon had quietly patented in 1998: **item-based collaborative filtering**. Flip the perspective — instead of finding similar _users_, find similar _items_. "Star Wars" and "Blade Runner" attract similar ratings from similar people; recommend one to fans of the other. This proved more scalable: items don't change their preferences, but users do. The Netflix Prize (2006–2009) showed that CF alone could get within 6% of the winning solution. Today, CF is the backbone of Spotify, YouTube, and TikTok recommendations. The core insight, unchanged since 1994: **your taste is encoded in your ratings history, and people with similar history will like similar things in the future**.
>
> **Where you are.** Chapter two of the Recommender Systems track. The popularity baseline from Ch.1 gave every user the same 10 movies — HR@10 ≈ 42%. Now you personalise: find users with similar taste (user-based CF) or find movies with similar rating patterns (item-based CF), and recommend accordingly. This is the first time FlixAI treats different users differently.
>
> **Notation.** $r_{ui}$ — rating by user $u$ on item $i$ (1–5, or missing if unrated); $\text{sim}(u, v)$ — cosine similarity between users $u$ and $v$; $\mathcal{N}_k(u)$ — the $k$ nearest neighbours of user $u$; $\bar{r}_u$ — mean rating of user $u$ across all rated items; $\hat{r}_{ui}$ — predicted rating for user $u$ on item $i$; $I_{uv}$ — set of items co-rated by both $u$ and $v$; $K$ — neighbourhood size hyperparameter.

---

## 0 · The Challenge

> **The mission**: FlixAI — >85% HR@10 across 5 constraints.

**What we know so far:**

- Ch.1: Popularity baseline = **42% HR@10** — everyone gets the same 10 movies
- MovieLens 100k: 943 users, 1,682 movies, 93.7% sparse
- Evaluation framework established (HR@10, NDCG@10, leave-one-out split)
- **But zero personalisation — a 20-year-old action fan and a 60-year-old romance lover see identical recommendations**

**What's blocking us:**
The popularity baseline treats User 1 (who only rates horror films) and User 2 (who only rates romance films) identically. Both see "The Shawshank Redemption" and "Pulp Fiction" at the top of their list. The system doesn't even read the ratings column — it just counts. Your VP of Product: _"If we're just showing everyone the popular movies, why do we need machine learning? We could do this in Excel."_

**What this chapter unlocks:**

- **User-based CF**: Find the K most similar users, weight their ratings → personalised prediction
- **Item-based CF**: Find items similar to what you already rated → stable + scalable
- **Explainability**: "Users who liked Star Wars also liked Blade Runner"
- **HR@10 jumps from 42% → ~68%** — 26 points from personalisation alone


## 1 · The Core Idea

The key observation driving collaborative filtering: you don't need to know anything about what a movie _is_ — genre, director, cast. You only need to know who rated it and how highly. If User 12 and User 47 have both rated the same 30 films similarly, they share a taste profile. What User 12 loved that User 47 hasn't seen yet is a strong recommendation signal. Scale this to 943 users and 100,000 ratings and the signal becomes powerful.

Two flavours exist, and they answer different questions:

- **User-Based CF**: "Find users like me — what did they watch that I haven't?" Personalised but expensive at scale: every new rating requires recomputing all pairwise user similarities.
- **Item-Based CF**: "Find movies like the ones I already rated." More stable: item similarity doesn't change when a new user joins. Precompute once, serve fast.

Both rely on cosine similarity applied to the sparse user-item matrix — no content features, no metadata, no genre labels. The prediction formula for user-based CF centers each user's ratings around their personal mean, so a user who always rates 4–5 stars doesn't artificially inflate the prediction for a user who rates 1–3:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

> **Optional depth:** Pearson correlation is equivalent to mean-centered cosine similarity on the co-rated items. Mean-centering removes user bias (generous vs harsh raters); cosine on co-rated items handles sparsity. See [MathUnderTheHood ch07](../../00-math-under-the-hood/ch07) for the derivation.


```mermaid
graph TD
    A["User A likes\nMovies 1,2,3"] --> C{"Find similar..."}
    B["User B likes\nMovies 1,2,4"] --> C
    C --> D["User-based CF:\nA and B are similar\nrecommend Movie 4 to A"]
    C --> E["Item-based CF:\nMovies 1,2 co-rated\nrecommend Movie 3\nto B's queue"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Imports
# 2. Compute `SEED` using `set_theme()`
# 3. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Load MovieLens 100k)
#
# Steps:
# 1. Load MovieLens 100k
# 2. Compute `ratings` using `read_csv()`
# 3. Compute `n_users` using `nunique()`
#
# Hint:
#    ratings = pd.read_csv(???)

In [ ]:
def leave_one_out_split(ratings_df):
    """
    TODO #3: Implement `leave_one_out_split()`.

    Steps:
    1. Leave-One-Out Split
    2. Process data

    Hint:
    ratings_sorted = ratings_df.sort_values(???)
    test = ratings_sorted.groupby(???)
    train = ratings_sorted.drop(???)

    Returns: train, test
    """
    raise NotImplementedError("TODO: implement leave_one_out_split()")

In [ ]:
def build_sparse_matrix(train_df, n_users, n_items):
    """
    TODO #4: Implement `build_sparse_matrix()`.

    Steps:
    1. Build Sparse User-Item Matrix
    2. Compute `R`

    Hint:
    # implement using the APIs described above

    Returns: csr_matrix((data, (row, col)), shape=...
    """
    raise NotImplementedError("TODO: implement build_sparse_matrix()")

In [ ]:
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #5: Implement `hit_rate_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement hit_rate_at_k()")


def ndcg_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #5: Implement `ndcg_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement ndcg_at_k()")

## 2 · User-Based Collaborative Filtering

The popularity baseline gave every user the same list. Here you fix that: find the $k$ users whose rating history most resembles the current user's, then use their ratings (weighted by similarity) to predict what the current user would give unrated films. Two users who agreed on 30 films are likely to agree on the 31st — the question is only how much weight to give each neighbour.

The mean-centering step matters. A user who always gives 4–5 stars and a user who always gives 1–2 stars can actually have very similar _taste preferences_ — they just calibrate their scale differently. By subtracting each user's mean before computing similarity, you compare deviations from personal baseline rather than raw scores.

Find the $k$ most similar users (cosine on mean-centered ratings) and predict:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

> **Optional depth:** The choice of $k$ trades bias for variance. Small $k$: personalised but noisy (few neighbours). Large $k$: stable but diluted (distant neighbours pollute the signal). The typical sweet spot for MovieLens 100k is $k = 20$–$50$.


In [ ]:
def user_based_cf(R_sparse, k_neighbors=50, n_recs=10):
    """
    TODO #6: Implement `user_based_cf()`.

    Steps:
    1. User-Based CF
    2. Call `ratings()` to produce the result
    3. Call `similarity()` to produce the result
    4. Call `argsort()` to produce the result
    5. Call `zeros()` to produce the result
    6. Call `argsort()` to produce the result
    7. Process data
    8. Compute `top_k_user_cf` using `CF()`
    9. Compute `hr_ucf` using `hit_rate_at_k()`

    Hint:
    R_dense = R_sparse.toarray(???)
    user_means = np.zeros(???)
    R_centered = R_dense.copy(???)
    top_neighbors = np.argsort(???)

    Returns: recommendations
    """
    raise NotImplementedError("TODO: implement user_based_cf()")

**Reflection — § 2 User-Based CF**

User-based CF achieves ~60% HR@10 — a substantial jump from the 42% popularity baseline. The personalisation is real: two users with opposite tastes now receive entirely different top-10 lists. But two problems emerge immediately:

1. **Sparsity**: With 93.7% of the matrix empty, most user pairs co-rate fewer than 5 movies. A cosine similarity computed on 3 shared films is statistically meaningless, yet the model uses it as if it were reliable.
2. **Scalability**: Computing pairwise user similarity requires an $O(m^2)$ pass. With 943 users that's 888,000 pairs — manageable now, but at 1M users it's 10^12 operations.

Item-based CF addresses both: items have more ratings per entity than users do, and item similarities are precomputable and stable (a movie's genre doesn't change when a new user joins).


## 3 · Item-Based Collaborative Filtering

The sparsity problem in user-CF is a data problem, not an algorithm problem. You can sidestep it by flipping the similarity axis: instead of finding users similar to you, find _items_ similar to the items you already rated, then score those similar items using your own ratings. The key advantage: items accumulate ratings from hundreds of users, making item-item similarities more statistically robust than user-user similarities in a sparse matrix.

The trade-off is interpretability: "Blade Runner is similar to Star Wars" is something you can explain to a user. "User 47 is similar to User 12" is not.

Item similarities are precomputed once — they don't change when a new user joins. For user $u$, the predicted score for unrated item $i$ is:

$$\hat{r}_{ui} = \frac{\sum_{j \in \mathcal{N}_k(i)} \text{sim}(i, j) \cdot r_{uj}}{\sum_{j \in \mathcal{N}_k(i)} |\text{sim}(i, j)|}$$

where $\mathcal{N}_k(i)$ is the set of the $k$ items most similar to $i$ that user $u$ has already rated.

> **Optional depth:** The denominator normalises by absolute similarity sum rather than raw sum — this prevents items with one very similar neighbour from dominating over items with many moderately similar neighbours. Without the absolute value, negative similarities (items that anti-correlate) could destructively cancel positive ones.


In [ ]:
def item_based_cf(R_sparse, k_neighbors=30, n_recs=10):
    """
    TODO #7: Implement `item_based_cf()`.

    Steps:
    1. Item-Based CF
    2. Call `toarray()` to produce the result
    3. Call `where()` to produce the result
    4. Call `zeros()` to produce the result
    5. Process data
    6. Call `argsort()` to produce the result
    7. Process data
    8. Compute `top_k_item_cf` using `CF()`
    9. Compute `hr_icf` using `hit_rate_at_k()`

    Hint:
    R_dense = R_sparse.toarray(???)
    rated_items = np.where(???)
    scores = np.zeros(???)
    top_k = np.argsort(???)

    Returns: recommendations
    """
    raise NotImplementedError("TODO: implement item_based_cf()")

In [ ]:
# TODO: Implement this cell
#  (Compare Methods)
#
# Steps:
# 1. Compare Methods
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `bar()`
# 4. Plot results -- call `suptitle()`
# 5. Call `to_string()` to produce the result
#
# Hint:
#    results = pd.DataFrame(???)
#    axes = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#  (Effect of k (Neighbors) on Item-Based CF)
#
# Steps:
# 1. Effect of k (Neighbors) on Item-Based CF
# 2. Call `item_based_cf()` to produce the result
# 3. Plot results -- call `subplots()`
# 4. Process data
#
# Hint:
#    ax = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#  (Visualise Item Similarity Matrix (Top-20 Movies))
#
# Steps:
# 1. Visualise Item Similarity Matrix (Top-20 Movies)
# 2. Aggregate data into `top_20_ids` -- use `groupby()`
# 3. Compute `movies_df` using `read_csv()`
# 4. Compute `sim_subset` using `ix_()`
# 5. Plot results -- call `subplots()`
#
# Hint:
#    top_20_ids = ratings.groupby(???)
#    movies_df = pd.read_csv(???)
#    titles = movies_df.set_index(???)
#    ax = plt.subplots(???)

## Progress Check

| #   | Constraint     | Target                | Ch.2 Status                           |
| --- | -------------- | --------------------- | ------------------------------------- |
| 1   | ACCURACY       | >85% HR@10            | ~68% (+26 from popularity!)           |
| 2   | COLD START     | New users/items       | Fails for new users                   |
| 3   | SCALABILITY    | 1M+ ratings           | O(n²) similarity                      |
| 4   | DIVERSITY      | Not just popular      | Better — neighbors have diverse taste |
| 5   | EXPLAINABILITY | "Because you liked X" | "Users like you also watched..."      |

**Bottom line**: 68% hit rate — personalisation jumps 26 points! But sparsity limits us. 93.7% of the matrix is empty.

**Next**: Ch.3 — Matrix Factorization → compress the sparse matrix into dense latent vectors.


## Exercises

**Exercise 1 — Pearson vs Cosine**
Implement Pearson correlation for user-based CF (center each user's ratings by their mean before computing cosine similarity). Compare HR@10 against raw cosine.

**Exercise 2 — Minimum Overlap Threshold**
Modify item-based CF to require at least `min_overlap=5` co-rated items before computing similarity. How does this affect HR@10?

**Exercise 3 — Hybrid: Popularity + CF**
For users with fewer than 20 ratings, fall back to the popularity baseline. For users with ≥20 ratings, use item-based CF. Compare HR@10 against pure item-CF.


In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold — Pearson vs Cosine)
#
# Steps:
# 1. Set up: Exercise 1 scaffold — Pearson vs Cosine
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold — Minimum Overlap Threshold)
#
# Steps:
# 1. Set up: Exercise 2 scaffold — Minimum Overlap Threshold
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold — Hybrid Popularity + CF)
#
# Steps:
# 1. Set up: Exercise 3 scaffold — Hybrid Popularity + CF
# 2. Process data
#
# Hint:
#    user_rating_counts = train.groupby(???)